# V7.2 — causal C on the matched metric

Both conditions use the exact same `logit(answer_A) - logit(answer_B)` measurement. Full residual interchange is the primary signed, unclipped causal truth; the two-J-Lens-direction subspace clamp is reported alongside it as a diagnostic.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PATH'] = '/home/jovyan/.local/bin:/home/jovyan/.npm-global/bin:' + os.environ['PATH']
os.environ['HF_HOME'] = '/home/jovyan/.cache/huggingface'
os.environ['HF_HUB_CACHE'] = '/home/jovyan/.cache/huggingface/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/jovyan/.cache/huggingface/hub'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print(ROOT)

/home/jovyan/j-space-thoughts


In [2]:
from src.causal_read_v7 import run_causal_stage_v7

def progress(completed, total, row):
    if completed == 1 or completed % 10 == 0 or completed == total:
        print(f'{completed}/{total}: {dict(row)}', flush=True)

summary = run_causal_stage_v7(progress=progress)
summary

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

1/70: {'pair_id': 'symmetric-000', 'engine_full_C': 0.9537275064267352, 'dashboard_full_C': 0.25, 'engine_subspace_C': 0.019280205655526992, 'dashboard_subspace_C': -0.002570694087403599}


10/70: {'pair_id': 'symmetric-017', 'engine_full_C': 0.9122807017543859, 'dashboard_full_C': 0.16979949874686717, 'engine_subspace_C': 0.25689223057644106, 'dashboard_subspace_C': 0.03070175438596491}


20/70: {'pair_id': 'symmetric-027', 'engine_full_C': 0.9125412541254125, 'dashboard_full_C': 0.09942244224422442, 'engine_subspace_C': 0.13696369636963696, 'dashboard_subspace_C': 0.018564356435643567}


30/70: {'pair_id': 'symmetric-038', 'engine_full_C': 0.8942598187311178, 'dashboard_full_C': 0.14501510574018128, 'engine_subspace_C': 0.15105740181268884, 'dashboard_subspace_C': 0.022658610271903322}


40/70: {'pair_id': 'symmetric-053', 'engine_full_C': 0.924187725631769, 'dashboard_full_C': 0.12996389891696752, 'engine_subspace_C': 0.22924187725631767, 'dashboard_subspace_C': 0.03489771359807461}


50/70: {'pair_id': 'symmetric-081', 'engine_full_C': 0.8716814159292036, 'dashboard_full_C': 0.11283185840707964, 'engine_subspace_C': 0.30530973451327437, 'dashboard_subspace_C': 0.01696165191740413}


60/70: {'pair_id': 'symmetric-095', 'engine_full_C': 0.931513903192585, 'dashboard_full_C': 0.08753861997940268, 'engine_subspace_C': 0.11071060762100927, 'dashboard_subspace_C': 0.01132852729145211}


70/70: {'pair_id': 'symmetric-105', 'engine_full_C': 0.958984375, 'dashboard_full_C': 0.06770833333333333, 'engine_subspace_C': 0.21484375, 'dashboard_subspace_C': 0.016276041666666668}


{'causal_path': '/home/jovyan/j-space-thoughts/results/v7/raw/causal_C_v7.json',
 'causal_sha256': 'b0373fa98466e2a1f084c12f696511e394007fbd1cccd02a76e86512f94e8fd3',
 'n_pairs': 70,
 'causal_sanity': {'n_pairs': 70,
  'primary_truth': 'full_residual',
  'signed_unclipped': True,
  'full_state_gate_thresholds': {'engine_abs_C_median_gt': 0.5,
   'dashboard_abs_C_median_lt': 0.1},
  'full_residual': {'engine_C_median': 0.9207558597942351,
   'engine_abs_C_median': 0.9207558597942351,
   'engine_C_min': 0.7857889237199582,
   'engine_C_max': 1.0120253164556963,
   'dashboard_C_median': 0.11485630282525375,
   'dashboard_abs_C_median': 0.11485630282525375,
   'dashboard_C_min': 0.05822784810126583,
   'dashboard_C_max': 0.2887139107611549,
   'engine_sharp_directional_disagreements': 0,
   'dashboard_sharp_directional_disagreements': 0},
  'jlens_two_concept_subspace': {'engine_C_median': 0.20861121726037013,
   'engine_abs_C_median': 0.20861121726037013,
   'engine_C_min': 0.009186351706

In [3]:
import json

artifact = json.loads(Path(summary['causal_path']).read_text())
assert artifact['signed_unclipped']
assert artifact['primary_truth'] == 'full_residual'
assert artifact['metric_contract']['engine'] == artifact['metric_contract']['dashboard']
assert all(row['same_metric_in_both_conditions'] for row in artifact['rows'])
assert all(row['engine']['full_residual']['signed_unclipped'] for row in artifact['rows'])
assert all(row['dashboard']['full_residual']['signed_unclipped'] for row in artifact['rows'])
assert all(row['engine']['jlens_two_concept_subspace']['signed_unclipped'] for row in artifact['rows'])
assert all(row['dashboard']['jlens_two_concept_subspace']['signed_unclipped'] for row in artifact['rows'])
assert all(
    row['engine']['full_residual']['T'] == row['engine']['jlens_two_concept_subspace']['T']
    for row in artifact['rows']
)
{
    'causal_sanity': artifact['causal_sanity'],
    'max_subspace_condition': max(
        row['engine']['jlens_two_concept_subspace']['subspace_gram_condition']
        for row in artifact['rows']
    ),
    'example': {
        'pair_id': artifact['rows'][0]['pair_id'],
        'metric': artifact['rows'][0]['metric'],
        'engine': artifact['rows'][0]['engine'],
        'dashboard': artifact['rows'][0]['dashboard'],
    },
}

{'causal_sanity': {'full_residual': {'dashboard_C_max': 0.2887139107611549,
   'dashboard_C_median': 0.11485630282525375,
   'dashboard_C_min': 0.05822784810126583,
   'dashboard_abs_C_median': 0.11485630282525375,
   'dashboard_sharp_directional_disagreements': 0,
   'engine_C_max': 1.0120253164556963,
   'engine_C_median': 0.9207558597942351,
   'engine_C_min': 0.7857889237199582,
   'engine_abs_C_median': 0.9207558597942351,
   'engine_sharp_directional_disagreements': 0},
  'full_state_dashboard_near_zero': False,
  'full_state_engine_large': True,
  'full_state_gate_thresholds': {'dashboard_abs_C_median_lt': 0.1,
   'engine_abs_C_median_gt': 0.5},
  'full_state_sanity_status': 'FAIL',
  'jlens_two_concept_subspace': {'dashboard_C_max': 0.05795314426633785,
   'dashboard_C_median': 0.02199364302415517,
   'dashboard_C_min': -0.004310344827586207,
   'dashboard_abs_C_median': 0.02199364302415517,
   'dashboard_sharp_directional_disagreements': 0,
   'engine_C_max': 0.487234042553191